# KIRAVO — Free Kaggle GPU Worker\nEnable T4 x2, then run the next cell.

In [ ]:
# KIRAVO — Free Kaggle GPU Worker
# Run this notebook once after enabling Kaggle T4 x2.
# The worker uses one T4 at a time and CPU/offload memory management.

!wget -q https://github.com/Iamkiranofficial/Kiravoo/raw/main/kaggle/kiravo_kaggle_worker.py -O /kaggle/working/kiravo_kaggle_worker.py

# Start the KIRAVO worker in the background.
!nohup python /kaggle/working/kiravo_kaggle_worker.py > /kaggle/working/kiravo-worker.log 2>&1 &

# Install Cloudflare's tunnel client.
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /kaggle/working/cloudflared
!chmod +x /kaggle/working/cloudflared

# Start a temporary public HTTPS tunnel.
!nohup /kaggle/working/cloudflared tunnel --url http://127.0.0.1:7860 --no-autoupdate > /kaggle/working/kiravo-tunnel.log 2>&1 &

import re, time
from pathlib import Path

for _ in range(30):
    text = Path('/kaggle/working/kiravo-tunnel.log').read_text(errors='ignore') if Path('/kaggle/working/kiravo-tunnel.log').exists() else ''
    match = re.search(r'https://[a-z0-9-]+\\.trycloudflare\\.com', text)
    if match:
        print('KIRAVO_WORKER_URL =', match.group(0))
        break
    time.sleep(2)
else:
    print('Tunnel URL not found yet. Run: !cat /kaggle/working/kiravo-tunnel.log')
